In [1]:
import matplotlib as plt
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import cv2 


In [2]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')#for face
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')#for Eyes

In [3]:
def get_cropped_image_if_2_eyes(image_path):
    import os
    img = cv2.imread(image_path)
    
    if img is None:
        print(f"Failed to load image: {image_path}")
        return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    for (x,y,w,h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = img[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        if len(eyes) >= 2:
            return roi_color
    
    return None

In [21]:
#path_to_data = r"D:\ImageClassificationProject\project dataset"
path_to_data = r"D:\ImageClassificationProject\project dataset2\project dataset"

path_to_cr_data = r"D:\ImageClassificationProject\project dataset2\project datasetcropped"

In [22]:
print(path_to_cr_data)

D:\ImageClassificationProject\project dataset2\project datasetcropped


In [23]:
import os
img_dirs = []
for entry in os.scandir(path_to_data):
    if entry.is_dir():
        img_dirs.append(entry.path)

In [24]:
import shutil
if os.path.exists(path_to_cr_data):
     shutil.rmtree(path_to_cr_data)
os.mkdir(path_to_cr_data)

In [25]:
cropped_image_dirs = [] #store cropped images diresctories
celebrity_file_names_dict = {} #store Name as key and thei files as value {Name1:[],Name2:[]...}

for img_dir in img_dirs:
    count = 1
    celebrity_name = img_dir.split('\\')[-1]
    print(celebrity_name)
    

abdul rehman
abood
bhavish
dr aftab


In [26]:
cropped_image_dirs = []
celebrity_file_names_dict = {}

for img_dir in img_dirs:
    count = 1
    celebrity_name = img_dir.split('\\')[-1]
    print(celebrity_name)
    
    celebrity_file_names_dict[celebrity_name] = []
    
    for entry in os.scandir(img_dir):
        roi_color = get_cropped_image_if_2_eyes(entry.path)
        if roi_color is not None:
            cropped_folder =os.path.join(path_to_cr_data, celebrity_name) #D:\ImageClassificationProject\project dataset\cropped\AbdulRehman
            if not os.path.exists(cropped_folder):
                os.makedirs(cropped_folder)
                cropped_image_dirs.append(cropped_folder)
                print("Generating cropped images in folder: ",cropped_folder)
                
            cropped_file_name = celebrity_name + str(count) + ".JPG"  #AbdulRehman1.JPG
            cropped_file_path = cropped_folder + "/" + cropped_file_name  #D:\ImageClassificationProject\project dataset\cropped\AbdulRehman/AbdulRehman1
            
            cv2.imwrite(cropped_file_path, roi_color)
            celebrity_file_names_dict[celebrity_name].append(cropped_file_path)
            count += 1

abdul rehman
Generating cropped images in folder:  D:\ImageClassificationProject\project dataset2\project datasetcropped\abdul rehman
abood
Generating cropped images in folder:  D:\ImageClassificationProject\project dataset2\project datasetcropped\abood
bhavish
Generating cropped images in folder:  D:\ImageClassificationProject\project dataset2\project datasetcropped\bhavish
dr aftab
Generating cropped images in folder:  D:\ImageClassificationProject\project dataset2\project datasetcropped\dr aftab


In [27]:
#Wavelet Transformation[feature Engineerng 
print('celebrityFileNameDict',celebrity_file_names_dict)

celebrityFileNameDict {'abdul rehman': ['D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman1.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman2.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman3.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman4.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman5.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman6.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman7.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul rehman8.JPG', 'D:\\ImageClassificationProject\\project dataset2\\project datasetcropped\\abdul rehman/abdul re

In [28]:
#Wavelet transformation
import pywt
def w2d(img, mode='haar', level=1):
    imArray = img
    #Datatype conversions
    #convert to grayscale
    imArray = cv2.cvtColor( imArray,cv2.COLOR_RGB2GRAY )
    #convert to float
    imArray =  np.float32(imArray)   
    imArray /= 255;
    # compute coefficients 
    coeffs=pywt.wavedec2(imArray, mode, level=level)
   # print('coeffs\n',coeffs)

    #Process Coefficients
    coeffs_H=list(coeffs) 
    #print('\n before coeffs\n',coeffs_H)
    coeffs_H[0] *= 0;
    #print(' after coeffs\n',coeffs)
 

    # reconstruction
    imArray_H=pywt.waverec2(coeffs_H, mode);
    imArray_H *= 255;
    imArray_H =  np.uint8(imArray_H)

    return imArray_H

In [29]:
class_dict={}
count=0
for celebrity_name in celebrity_file_names_dict.keys():
    print(celebrity_name)
    class_dict[celebrity_name]=count
    count+=1

print(class_dict)

abdul rehman
abood
bhavish
dr aftab
{'abdul rehman': 0, 'abood': 1, 'bhavish': 2, 'dr aftab': 3}


In [30]:
X=[]
y=[]
for celebrity_name,TrainingFilesPath in celebrity_file_names_dict.items():
    for training_image in TrainingFilesPath:
        img=cv2.imread(training_image)
        scaled_raw_img=cv2.resize(img,(32,32))
        img_har=w2d(img,'db1',5)
        scaled_img_har=cv2.resize(img_har,(32,32))
        combined_imgs=np.vstack((scaled_raw_img.reshape(32*32*3,1),scaled_img_har.reshape(32*32,1)))
        X.append(combined_imgs)
        y.append(class_dict[celebrity_name])
        
        
        

In [31]:
len(X[0])

4096

In [32]:
X[0]

array([[214],
       [216],
       [226],
       ...,
       [246],
       [  2],
       [111]], dtype=uint8)

In [33]:
y[0]

0

In [34]:
X=np.array(X).reshape(len(X),4096).astype(float)
X.shape

(85, 4096)

In [35]:
#Train the model
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42,test_size=0.2)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=50,random_state=0)

pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC(kernel = 'rbf', C = 10))])


In [38]:
pipe.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()), ('svc', SVC(C=10))])

In [39]:
print(classification_report(y_test,pipe.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.71      0.83        14
           1       1.00      1.00      1.00         9
           2       1.00      0.79      0.88        14
           3       0.65      1.00      0.79        13

    accuracy                           0.86        50
   macro avg       0.91      0.88      0.88        50
weighted avg       0.91      0.86      0.86        50



In [40]:
train_accuracy = pipe.score(X_train, y_train)
test_accuracy = pipe.score(X_test, y_test)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
#print('Training Accuracy',train_accuracy)
#print('Test Accuracy',train_accuracy)

Training Accuracy: 100.00%
Test Accuracy: 86.00%


In [41]:
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(n_estimators=340,n_jobs=4,random_state=10)

In [42]:
model.fit(X_train,y_train)

RandomForestClassifier(n_estimators=340, n_jobs=4, random_state=10)

In [48]:
print(f"train_score_accuracy: {model.score(X_train, y_train) * 100:.2f}%")
print(f"test_score_accuracy: {model.score(X_test, y_test) * 100:.2f}%")


train_score_accuracy: 100.00%
test_score_accuracy: 90.00%
